In [1]:
# %% Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# %% Load data
data = pd.read_csv("Football_Dataset_2015_2025.csv")

# Drop columns not useful
data = data.drop(["Date"], axis=1)

# Encode categorical variables
labels_cols = ["Competition", "Home Team", "Away Team"]
encoder = OrdinalEncoder()
data[labels_cols] = encoder.fit_transform(data[labels_cols])

# --- Drop outcome leakage ---
# Do NOT use Home Goals / Away Goals as predictors
drop_cols = ["Winner", "Year", "Home Goals", "Away Goals"]
X = data.drop(drop_cols, axis=1)
y = data["Winner"]

# --- Feature engineering: differences between teams ---
X["Possession_Diff"] = data["Possession % (Home)"] - data["Possession % (Away)"]
X["Shots_Diff"] = data["Shots (Home)"] - data["Shots (Away)"]
X["Corners_Diff"] = data["Corners (Home)"] - data["Corners (Away)"]
X["Fouls_Diff"] = data["Fouls (Home)"] - data["Fouls (Away)"]

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Scale numeric features ---
num_cols = ['Possession % (Home)', 'Possession % (Away)', 
            'Shots (Home)', 'Shots (Away)', 
            'Corners (Home)', 'Corners (Away)', 
            'Fouls (Home)', 'Fouls (Away)', 
            'Possession_Diff', 'Shots_Diff', 'Corners_Diff', 'Fouls_Diff']

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# --- Logistic Regression with regularization tuning ---
model = LogisticRegression(max_iter=500, multi_class="multinomial", solver="lbfgs", C=2.0)
model.fit(X_train, y_train)

# --- Predictions ---
y_pred = model.predict(X_test)

# --- Evaluation ---
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.34

Classification Report:
               precision    recall  f1-score   support

   Away Team       0.38      0.34      0.36       219
        Draw       0.32      0.32      0.32       190
   Home Team       0.32      0.37      0.34       191

    accuracy                           0.34       600
   macro avg       0.34      0.34      0.34       600
weighted avg       0.34      0.34      0.34       600


Confusion Matrix:
 [[74 62 83]
 [66 60 64]
 [55 66 70]]


c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [2]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 2, 5],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [200, 500]
}
grid = GridSearchCV(LogisticRegression(multi_class="multinomial"), param_grid, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)


c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\ADMIN\AppData\Local\Programs\Python\P

Best params: {'C': 2, 'max_iter': 200, 'solver': 'saga'}


c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [3]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.34

Classification Report:
               precision    recall  f1-score   support

   Away Team       0.38      0.34      0.36       219
        Draw       0.32      0.32      0.32       190
   Home Team       0.32      0.37      0.34       191

    accuracy                           0.34       600
   macro avg       0.34      0.34      0.34       600
weighted avg       0.34      0.34      0.34       600


Confusion Matrix:
 [[74 62 83]
 [65 60 65]
 [55 66 70]]
